# 3.1 Tickers
The goal is to solve the problems of the Polygon ticker lists in the introduction. Before we do that we will download the ticker list for all days from Polygon and store them into the map <code>tickers</code>.

In [1]:
from massive.rest import RESTClient
from datetime import datetime, date, time, timedelta
from pytz import timezone
from functools import lru_cache
from times import get_market_dates, first_trading_date_after, last_trading_date_before, first_trading_date_after_equal, last_trading_date_before_equal
import os
import pytz
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import mplfinance as mpf
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import glob
from dotenv import load_dotenv 

# Now load variables and define constants
load_dotenv() # to load variables from .env

ACCESS_KEY_ID = os.environ.get("ACCESS_KEY_ID")
if not ACCESS_KEY_ID:
    raise ValueError("No API key found. Please set ACCESS_KEY_ID in your .env file.")

API_KEY = os.environ.get("MASSIVE_API_KEY")
if not API_KEY:
    raise ValueError("No API key found. Please set MASSIVE_API_KEY in your .env file.")

client = RESTClient(api_key=API_KEY)

POLYGON_DATA_PATH = "../data/polygon/"

START_DATE = date(2016, 5, 24)
END_DATE = date(2023, 5, 21)

## VFS ticker exists in raw files! 

In [19]:
# List to hold filtered DataFrames
vfs_frames = []
first_found = False          # Flag to track first file with 'VFS'

files = os.listdir(POLYGON_DATA_PATH + 'test/tickers')

for file in files:
    df = pd.read_csv(POLYGON_DATA_PATH + f'test/tickers/{file}', index_col=0)
    
    # Filter rows where ticker == 'VFS'
    filtered = df[df['ticker'] == 'VFS']
    
    # Append if not empty
    if not filtered.empty:
        vfs_frames.append(filtered)
        
        # Print the first such filename only once
        if not first_found:
            print(f"First file containing 'VFS': {file}")
            first_found = True

# Combine all filtered data into one DataFrame
if vfs_frames:
    vfs_df = pd.concat(vfs_frames, ignore_index=True)
else:
    vfs_df = pd.DataFrame()  # Empty DataFrame if no matches

print(f"Total rows with ticker 'VFS': {len(vfs_df)}")

First file containing 'VFS': 2023-10-26.csv
Total rows with ticker 'VFS': 172


In [ ]:
# VFS "list_date": "2023-08-15" 
# https://massive.com/docs/rest/stocks/tickers/ticker-overview

# vfs_df.sort_values(['last_updated_utc'], inplace=True)

vfs_df

,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type
106,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2024-12-03T20:05:57.474722Z,1913510.0,NaN,CS
97,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2024-12-03T20:05:58.868482Z,1913510.0,NaN,CS
57,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2024-12-03T20:05:59.171882Z,1913510.0,NaN,CS
80,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2024-12-03T20:06:01.646507Z,1913510.0,NaN,CS
73,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2024-12-03T20:06:02.251002Z,1913510.0,NaN,CS
...,...,...,...,...,...,...,...,...
168,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2025-08-13T14:43:34.208948Z,1913510.0,NaN,CS
110,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2025-08-13T14:44:32.053137Z,1913510.0,NaN,CS
119,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2025-08-13T14:44:47.549244Z,1913510.0,NaN,CS
24,VFS,VinFast Auto Ltd. Ordinary Shares,True,NaN,2025-08-13T14:45:43.157237Z,1913510.0,NaN,CS


#### VFS remains an Active ticker so it should have come up later! Why didnt it?

In [4]:
vfs_df[vfs_df['active'] == False]

,ticker,name,active,delisted_utc,last_updated_utc,cik,composite_figi,type


# 3.2 Building the tickers loop
Now we can finally create our ticker list, which includes all tickers. The process involves looping over all Polygon ticker lists and updating our own one. First some notation: T is our ticker list that we iteratively update using Polygons ticker list. P(i) is the Polygon ticker list from day *i*. 

1. On day 1, our ticker list is the same as the one from Polygon, but with some extra columns. We create a column <code>start_date</code> which is day 1 and <code>end_date</code> with is empty. We are only interested in stocks that were active on that day.
2. For all *i = 2 ... n* days, for the active stocks:
    * **Delistings**: The stocks that are in T but not in P(i) are the stocks that are removed by Polygon (e.g. FB). For these tickers we set the <code>end_date</code> in T to day *i-1*. 
    * **New listings**: The stocks that are in P(i) but not in T are the new listings. We will append the new stock to T and set the start_date to day *i*.
    * **Everything else**: The stocks that are both in P(i) and T are the stocks that 'continue their listings'. We do nothing.

Two tickers are the 'same' if all fields except <code>last_updated_utc</code> or <code>delisted_utc</code> are the same.

For testing, we will start with 2022-06-08 and update to 2022-06-09. Both FB and META should then be included with the correct start and end dates. The start and end date of FB should be 2022-06-08 and the start date of META should be 2022-06-09. The end date of META should be empty.

"The stock ticker for Meta Platforms, Inc. officially changed from FB to META on June 9, 2022 at the start of the trading day / Open".

Massive confirms this too! https://massive.com/docs/rest/stocks/corporate-actions/ticker-events (Experimental Endpoint though! So don't go by this yet!)

SO WHY IS DATA DIFFERENT! AND WHY DIFFERENT NOW AND NOT A YEAR AGO!

Example of issue:

META mega-cap started trading on June 9, 2022 at the start of the trading day / Open. OHLCV data only exists for META from 2022-06-09

FB mega-cap stopped trading on June 8, 2022. OHLCV data only exists for FB till 2022-06-08.

https://massive.com/docs/rest/stocks/aggregates/custom-bars endpoint confirms that OHLCV data only exists for META from 2022-06-09, and only exists for FB till 2022-06-08.

So this confirms that Massive changed things and now shows the Old ticker for 1 more day even when it's the New ticker trading on that date and not the old! 
(maybe is now using that Ticker Change experimental endpoint that shows 2022-06-09 as Ticker change date)

And if Massive is doing this for a Mega-cap like META, it's doing it for ALL stocks, so we can be rest assured that we can make this change. 
Besides, it's better to make this change a day earlier than the Event date. 

So we need to use day1-1 for the end_date.

**Basically, the OHLCV endpoint gives no data for FB and should be the one we need to be using to check End-date!**



# 3.3 The tickers loop
The ticker lists of 2009-10-29, 2010-03-30 and 2010-03-31 seem incomplete. We will simply copy the ticker list the trading day before to avoid errors.

Putting it all in a loop gives the following code. We save the results to <code>tickers_v1.csv</code>.

In [ ]:
# --- CONVERT DATE RANGE TO PANDAS TIMESTAMP ---
# Changed: Use pd.Timestamp instead of date to avoid dtype issues
START_DATE = pd.Timestamp(date(2016, 5, 24))
END_DATE = pd.Timestamp(date(2023, 5, 24))

# Assumed existing functions (they must return date-like objects)
# We'll convert their results to Timestamp as well
market_dates_raw = get_market_dates()  # list of datetime.date or Timestamp
market_days = [pd.Timestamp(d) for d in market_dates_raw]  # force Timestamp

# Adjust to trading days – convert to date for comparison, then back to Timestamp
START_DATE = pd.Timestamp(first_trading_date_after_equal(START_DATE.date()))
END_DATE = pd.Timestamp(last_trading_date_before_equal(END_DATE.date()))

print(START_DATE)
print(END_DATE)
print(market_days[:5], market_days[-5:])

# Initialize our_tickers as None
our_tickers = None

for day in market_days:
    if day == START_DATE:
        # At the start, our ticker list is the same as polygon.
        our_tickers = pd.read_csv(
            POLYGON_DATA_PATH + f"test/tickers/{START_DATE.date().isoformat()}.csv",
            index_col=0,
            keep_default_na=False,
            na_values=['#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan', '1.#IND', '1.#QNAN', '<NA>', 'N/A', 'NULL', 'NaN', 'None', 'n/a', 'nan', 'null']
        )  # There is a stock named 'NA'. Avoid pandas treating it as N/A.
        our_tickers = our_tickers[["ticker", "name", "active", "cik", "composite_figi", "type"]]
        our_tickers = our_tickers[our_tickers["active"]]

        # Sometimes a cik can be empty. This prevents merge function from working.
        our_tickers['cik'] = our_tickers['cik'].replace('', np.nan)
        our_tickers['cik'] = our_tickers['cik'].astype(float)

        our_tickers.reset_index(inplace=True, drop=True)

        # Initialize tickers_all with Timestamp start_date
        our_tickers["start_date"] = START_DATE  # pd.Timestamp
        our_tickers["end_date"] = pd.NaT        # pandas datetime64[ns] compatible

    elif day > START_DATE and day <= END_DATE:
        # Get new ticker list to update ours
        tickers_day_i = pd.read_csv(
            POLYGON_DATA_PATH + f"test/tickers/{day.date().isoformat()}.csv",
            index_col=0,
            keep_default_na=False,
            na_values=['#N/A', '#N/A N/A', '#NA', '-1.#IND', '-1.#QNAN', '-NaN', '-nan', '1.#IND', '1.#QNAN', '<NA>', 'N/A', 'NULL', 'NaN', 'None', 'n/a', 'nan', 'null']
        )
        tickers_day_i['cik'] = tickers_day_i['cik'].replace('', np.nan)
        tickers_day_i['cik'] = tickers_day_i['cik'].astype(float)

        tickers_day_i.sort_values(['last_updated_utc'], inplace=True)

        tickers_day_i = tickers_day_i[["ticker", "name", "active", "cik", "composite_figi", "type"]]
        tickers_day_i = tickers_day_i[tickers_day_i["active"]]
        tickers_day_i.reset_index(inplace=True, drop=True)

        # Remove duplicates
        duplicated = tickers_day_i[tickers_day_i["ticker"].duplicated(keep=False)]
        indices_to_remove = duplicated["ticker"].duplicated(keep='last')
        tickers_day_i.drop(list(indices_to_remove[indices_to_remove].index), inplace=True)
        tickers_day_i.reset_index(drop=True, inplace=True)

        # Preliminary check: no duplicates
        if our_tickers.duplicated().any():
            raise Exception("There are duplicates in our_tickers!")
        if tickers_day_i.duplicated().any():
            raise Exception("There are duplicates in tickers_day_i!")

        # DELISTINGS
        indicator_delisted = our_tickers[["ticker", "name", "active", "cik", "composite_figi", "type"]].merge(
            tickers_day_i[["ticker", "name", "active", "cik", "composite_figi", "type"]],
            on=["ticker", "name", "active", "cik", "composite_figi", "type"],
            how='left', indicator=True
        )

        indicator_delisted['_merge'] = np.where(
            our_tickers["active"],
            indicator_delisted['_merge'],
            "both"
        )  # ERROR FIX: already inactive stocks should not be delisted again

        indicator_delisted = indicator_delisted["_merge"]
        delisted_tickers = our_tickers[indicator_delisted == "left_only"]

        # NEW LISTINGS
        indicator_new = tickers_day_i[["ticker", "name", "active", "cik", "composite_figi", "type"]].merge(
            our_tickers[["ticker", "name", "active", "cik", "composite_figi", "type"]],
            on=["ticker", "name", "active", "cik", "composite_figi", "type"],
            how='left', indicator=True
        )
        indicator_new = indicator_new["_merge"]
        new_tickers = tickers_day_i[indicator_new == "left_only"]

        # PROCESS DELISTINGS
        # Get previous trading day (already a pd.Timestamp)
        idx = market_days.index(day)
        previous_day = market_days[idx - 1]  # pd.Timestamp

        # Assign end_date as Timestamp (compatible with datetime64[ns])
        our_tickers.loc[indicator_delisted == "left_only", "end_date"] = previous_day
        our_tickers.loc[indicator_delisted == "left_only", "active"] = False

        # PROCESS NEW LISTINGS
        our_tickers = pd.concat([our_tickers, new_tickers], ignore_index=True)
        # Fill missing start_date with current day (pd.Timestamp)
        our_tickers['start_date'] = our_tickers['start_date'].fillna(value=day)

        # Final checks
        if our_tickers[["ticker", "name", "active", "type", "start_date"]].isnull().values.any():
            raise Exception("There are missing values in required columns.")

        print(f'{day.date().isoformat()}: Amount of stocks {len(our_tickers)}')

        # Finalize if last day
        if day == END_DATE:
            our_tickers["end_date"] = our_tickers["end_date"].fillna(value=END_DATE)
            our_tickers = our_tickers.sort_values(by=["ticker", "end_date"]).reset_index(drop=True)
            our_tickers[["ticker", "name", "active", "start_date", "end_date", "type", "cik", "composite_figi"]] \
                .to_csv("../data/tickers_vfs.csv")

2016-05-24 00:00:00
2023-05-24 00:00:00
2016-05-25: Amount of stocks 5759
2016-05-26: Amount of stocks 5769
2016-05-27: Amount of stocks 5775
2016-05-31: Amount of stocks 5784
2016-06-01: Amount of stocks 5788
2016-06-02: Amount of stocks 5798
2016-06-03: Amount of stocks 5804
2016-06-06: Amount of stocks 5806
2016-06-07: Amount of stocks 5808
2016-06-08: Amount of stocks 5809
2016-06-09: Amount of stocks 5818
2016-06-10: Amount of stocks 5826
2016-06-13: Amount of stocks 5835
2016-06-14: Amount of stocks 5842
2016-06-15: Amount of stocks 5848
2016-06-16: Amount of stocks 5854
2016-06-17: Amount of stocks 5858
2016-06-20: Amount of stocks 5864
2016-06-21: Amount of stocks 5874
2016-06-22: Amount of stocks 5883
2016-06-23: Amount of stocks 5890
2016-06-24: Amount of stocks 5892
2016-06-27: Amount of stocks 5898
2016-06-28: Amount of stocks 5910
2016-06-29: Amount of stocks 5913
2016-06-30: Amount of stocks 5917
2016-07-01: Amount of stocks 5924
2016-07-05: Amount of stocks 5933
2016-07-

We also create a function to retrieve the ticker list.

In [34]:
def get_tickers(v=5):
    """
    Retrieve the ticker list. Default is 5.
    """
    tickers = pd.read_csv(
        f"../data/tickers_v{v}.csv",
        parse_dates=["start_date", "end_date"],
        index_col=0,
        keep_default_na=False,
        na_values=["#N/A","#N/AN/A","#NA","-1.#IND","-1.#QNAN","-NaN","-nan","1.#IND","1.#QNAN","<NA>","N/A","NULL","NaN","None","n/a","nan","null",],
    )
    tickers["start_date"] = pd.to_datetime(tickers["start_date"]).dt.date
    tickers["end_date"] = pd.to_datetime(tickers["end_date"]).dt.date

    # This will only be applied in future notebooks.
    if tickers.columns.isin(["start_data", "end_data"]).any():
        tickers["start_data"] = pd.to_datetime(tickers["start_data"]).dt.date
        tickers["end_data"] = pd.to_datetime(tickers["end_data"]).dt.date

    return tickers

In [ ]:
tickers_v1 = get_tickers(1)

# VFS missing!

In [ ]:
tickers_v1[tickers_v1["ticker"] == "VFS"]

In [ ]:
tickers_v1[tickers_v1["ticker"] == "FB"]

,ticker,name,active,start_date,end_date,type,cik,composite_figi
10551,FB,FACEBOOK INC CL A COM STK (DE),False,2016-05-24,2016-10-24,,1326801.0,BBG000MM2P62
10552,FB,"Facebook, Inc. Class A",False,2016-10-25,2019-09-23,CS,1326801.0,BBG000MM2P62
10553,FB,"Facebook, Inc. Class A",False,2019-09-24,2019-09-24,CS,,BBG000MM2P62
10554,FB,"Facebook, Inc. Class A",False,2019-09-25,2021-10-29,CS,1326801.0,BBG000MM2P62
10555,FB,"Meta Platforms, Inc. Class A Common Stock",False,2021-11-01,2022-06-09,CS,1326801.0,BBG000MM2P62


In [36]:
tickers_v1[tickers_v1["ticker"] == "META"]

,ticker,name,active,start_date,end_date,type,cik,composite_figi
17928,META,"Meta Platforms, Inc. Class A Common Stock",True,2022-06-09,2023-05-24,CS,1326801.0,BBG000MM2P62


In [37]:
print(len(tickers_v1))

31316
